In [14]:
import json
import os

import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from openai import OpenAI
from dotenv import load_dotenv

from utils.data import edinet_to_industry_map, all_securities, jp_500
from utils.datetime import date_string_to_quarter
from utils.edinet_api import get_doc_name
from utils.ELO import EloRatingSystem, get_num_games, generate_random_pdf_pair
from utils.signal import get_winner, get_stock_code_from_path

from utils.datapath import (
    documents_path,
    edinet_codes_path,
    docs_metadata_path,
    industry_elo_signals_path
)

In [15]:
load_dotenv()
XAI_API_KEY = os.getenv("XAI_API_KEY")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

grok_client = OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)

deepseek_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

In [16]:
df = pd.read_excel(edinet_codes_path)

with open(docs_metadata_path) as f:
    docs_metadata = json.load(f)

In [17]:
top_security_codes = list(set([code for quarter, codes in jp_500.items() for code in codes]))
top_securities = [security for security in all_securities if security["code"] in top_security_codes]
top_securities_edinet = [security["edinet_code"] for security in top_securities]
len(top_securities)

740

In [18]:
filtered_doc_metadata = [doc for doc in docs_metadata if doc["edinetCode"] in top_securities_edinet]
len(filtered_doc_metadata)

20175

In [19]:
industry_quarterly_docs = defaultdict(lambda: defaultdict(list))

for doc in filtered_doc_metadata:
    industry = edinet_to_industry_map.get(doc['edinetCode'], "na")
    period_end_quater = date_string_to_quarter(doc["periodEnd"])

    industry_path = os.path.join(documents_path, industry)
    quarter_path = os.path.join(industry_path, period_end_quater)

    save_name = get_doc_name(doc)
    output_path = os.path.join(quarter_path, save_name)

    industry_quarterly_docs[industry][period_end_quater].append(output_path)

In [24]:
num_requests = 0
industry_quarterly_signals = {}

for industry, quarterly_docs in tqdm(industry_quarterly_docs.items()):
    elo_system = EloRatingSystem(ratings={})
    quarterly_signals = {}

    for quarter, docs in quarterly_docs.items():
        for pdf1_path, pdf2_path in generate_random_pdf_pair(
            docs, get_num_games(len(docs))
        ):
            winner = get_winner(pdf1_path, pdf2_path)
            num_requests += 1
            companyA_code, companyB_code = [
                get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
            ]
            elo_system.update_ratings(companyA_code, companyB_code, winner)

        quarterly_signals[quarter] = elo_system.get_all_ratings()
    
    industry_quarterly_signals[industry] = quarterly_signals


100%|██████████| 33/33 [00:01<00:00, 24.16it/s]


In [31]:
# request_count

# Cost Analysis
model_input_prices = {
    "grok-3-mini": 0.3,
    "grok-3": 3,
    "deepseek-chat": 0.27,
    "deepseek-chat-discount": 0.135,
    "deepseek-reasoner": 0.55,
    "deepseek-reasoner-discount": 0.135,
}

model = "grok-3-mini"
num_input_token_per_request = 75000
price_per_million_input_token = model_input_prices[model] # US dollars 
price_per_request = (num_input_token_per_request / 1e6) * price_per_million_input_token
total_price = price_per_request * num_requests
print(f"Total cost per run: ${total_price:.2f}")

Total cost per run: $7736.26


In [8]:
# quarterly_signals should equal industry_quarterly_signals[industry]
# industry_quarterly_signals["Marine Transportation"]

In [ ]:
# Industry wise elo - DONE
# Industry wise portfolio backtest - DONE

# But that's not the important point
# Get to the api calls!!!!!!!!!

In [10]:
# Industry wise elo

with open(industry_elo_signals_path, 'w', encoding='utf-8') as f:
    json.dump(industry_quarterly_signals, f)

In [ ]:
# Deepseek >50% off from 12:30 to 8:30 am HKT